In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time
import os

In [ ]:
def trenuj_resnet(data_dir, num_classes, epochs=5, batch_size=8, lr=0.001, save_name='wytrenowany_resnet.pth'):
    """
    Funkcja do trenowania modelu ResNet-18.
    Parametry:
    - data_dir: ścieżka do folderu 'train' ze zdjęciami
    - num_classes: liczba klas
    - epochs: ile razy model ma przejrzeć dane
    - batch_size: ile zdjęć naraz przetwarza model
    - lr: learning rate
    - save_name: nazwa pliku końcowego
    """
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"--- START TRENINGU ---")
    print(f"Urządzenie: {device} | Klasy: {num_classes} | Epoki: {epochs}")

    # 1. Transformacje i ładowanie danych
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    if not os.path.exists(data_dir):
        print(f"BŁĄD: Folder {data_dir} nie istnieje!")
        return

    dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    dataset_size = len(dataset)
    print(f"Załadowano {dataset_size} zdjęć.")

    # 2. Budowa modelu
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 3. Pętla treningowa
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        running_corrects = 0
        
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        epoch_loss = running_loss / dataset_size
        epoch_acc = running_corrects.double() / dataset_size
        print(f'Epoka {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Czas: {time.time()-start_time:.0f}s')

    # 4. Zapis
    torch.save(model.state_dict(), save_name)
    print(f"Zapisano model jako: {save_name}\n")
    return model   



In [7]:
model_eksperyment_1 = trenuj_resnet(
    data_dir = './testy_nzal/FunnyBirds/train',
    num_classes = 4,
    epochs = 20,
    batch_size= 32,
    save_name = 'model_hierarchia_nzal.pth'
)

--- START TRENINGU ---
Urządzenie: cpu | Klasy: 4 | Epoki: 20
Załadowano 100 zdjęć.
Epoka 1/20 | Loss: 1.2769 | Acc: 0.4200 | Czas: 4s
Epoka 2/20 | Loss: 0.7541 | Acc: 0.7100 | Czas: 4s
Epoka 3/20 | Loss: 0.7049 | Acc: 0.7400 | Czas: 3s
Epoka 4/20 | Loss: 0.2362 | Acc: 0.9300 | Czas: 4s
Epoka 5/20 | Loss: 0.3204 | Acc: 0.8900 | Czas: 4s
Epoka 6/20 | Loss: 0.2560 | Acc: 0.9100 | Czas: 9s
Epoka 7/20 | Loss: 0.2345 | Acc: 0.9000 | Czas: 12s
Epoka 8/20 | Loss: 0.1205 | Acc: 0.9800 | Czas: 12s
Epoka 9/20 | Loss: 0.0811 | Acc: 0.9900 | Czas: 13s
Epoka 10/20 | Loss: 0.0673 | Acc: 0.9800 | Czas: 12s
Epoka 11/20 | Loss: 0.0607 | Acc: 0.9900 | Czas: 12s
Epoka 12/20 | Loss: 0.1468 | Acc: 0.9600 | Czas: 11s
Epoka 13/20 | Loss: 0.0282 | Acc: 1.0000 | Czas: 3s
Epoka 14/20 | Loss: 0.1100 | Acc: 0.9600 | Czas: 3s
Epoka 15/20 | Loss: 0.0549 | Acc: 0.9900 | Czas: 3s
Epoka 16/20 | Loss: 0.0376 | Acc: 1.0000 | Czas: 4s
Epoka 17/20 | Loss: 0.0681 | Acc: 0.9900 | Czas: 4s
Epoka 18/20 | Loss: 0.3342 | Acc: 0

In [8]:
model_eksperyment_2 = trenuj_resnet(
    data_dir = './testy_hier/FunnyBirds/train',
    num_classes = 3,
    epochs = 20,
    batch_size= 32,
    save_name = 'model_hierarchia_hier.pth'
)

--- START TRENINGU ---
Urządzenie: cpu | Klasy: 3 | Epoki: 20
Załadowano 75 zdjęć.
Epoka 1/20 | Loss: 1.1185 | Acc: 0.3600 | Czas: 4s
Epoka 2/20 | Loss: 0.3093 | Acc: 0.8800 | Czas: 3s
Epoka 3/20 | Loss: 0.1431 | Acc: 0.9333 | Czas: 3s
Epoka 4/20 | Loss: 0.1374 | Acc: 0.9333 | Czas: 3s
Epoka 5/20 | Loss: 0.0443 | Acc: 1.0000 | Czas: 3s
Epoka 6/20 | Loss: 0.0179 | Acc: 1.0000 | Czas: 3s
Epoka 7/20 | Loss: 0.0113 | Acc: 1.0000 | Czas: 3s
Epoka 8/20 | Loss: 0.0321 | Acc: 0.9867 | Czas: 3s
Epoka 9/20 | Loss: 0.0167 | Acc: 1.0000 | Czas: 3s
Epoka 10/20 | Loss: 0.0135 | Acc: 1.0000 | Czas: 3s
Epoka 11/20 | Loss: 0.0531 | Acc: 0.9867 | Czas: 3s
Epoka 12/20 | Loss: 0.0039 | Acc: 1.0000 | Czas: 3s
Epoka 13/20 | Loss: 0.0024 | Acc: 1.0000 | Czas: 3s
Epoka 14/20 | Loss: 0.0037 | Acc: 1.0000 | Czas: 3s
Epoka 15/20 | Loss: 0.0056 | Acc: 1.0000 | Czas: 3s
Epoka 16/20 | Loss: 0.0026 | Acc: 1.0000 | Czas: 3s
Epoka 17/20 | Loss: 0.0010 | Acc: 1.0000 | Czas: 3s
Epoka 18/20 | Loss: 0.1073 | Acc: 0.9467 |

In [9]:
model_eksperyment_3 = trenuj_resnet(
    data_dir = './testy_ABC/FunnyBirds/train',
    num_classes = 3,
    epochs = 20,
    batch_size= 32,
    save_name = 'model_hierarchia_ABC.pth'
)

--- START TRENINGU ---
Urządzenie: cpu | Klasy: 3 | Epoki: 20
Załadowano 30 zdjęć.
Epoka 1/20 | Loss: 1.2189 | Acc: 0.3667 | Czas: 2s
Epoka 2/20 | Loss: 0.0780 | Acc: 1.0000 | Czas: 1s
Epoka 3/20 | Loss: 0.0128 | Acc: 1.0000 | Czas: 1s
Epoka 4/20 | Loss: 0.0032 | Acc: 1.0000 | Czas: 1s
Epoka 5/20 | Loss: 0.0017 | Acc: 1.0000 | Czas: 1s
Epoka 6/20 | Loss: 0.0011 | Acc: 1.0000 | Czas: 1s
Epoka 7/20 | Loss: 0.0007 | Acc: 1.0000 | Czas: 1s
Epoka 8/20 | Loss: 0.0005 | Acc: 1.0000 | Czas: 1s
Epoka 9/20 | Loss: 0.0004 | Acc: 1.0000 | Czas: 1s
Epoka 10/20 | Loss: 0.0003 | Acc: 1.0000 | Czas: 1s
Epoka 11/20 | Loss: 0.0002 | Acc: 1.0000 | Czas: 1s
Epoka 12/20 | Loss: 0.0002 | Acc: 1.0000 | Czas: 1s
Epoka 13/20 | Loss: 0.0002 | Acc: 1.0000 | Czas: 1s
Epoka 14/20 | Loss: 0.0002 | Acc: 1.0000 | Czas: 1s
Epoka 15/20 | Loss: 0.0002 | Acc: 1.0000 | Czas: 1s
Epoka 16/20 | Loss: 0.0001 | Acc: 1.0000 | Czas: 1s
Epoka 17/20 | Loss: 0.0001 | Acc: 1.0000 | Czas: 1s
Epoka 18/20 | Loss: 0.0001 | Acc: 1.0000 |